### Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ All imports successful")

✅ All imports successful


### Database Connection 

In [2]:
# Define database path
db_path = Path(r'D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\data\ETH.db')

if not db_path.exists():
    print(f"❌ Database not found at: {db_path}")
    # Try relative path
    db_path = Path('../../../../data/ETH.db').resolve()
    if not db_path.exists():
        print("❌ Database still not found!")
        print(f"Current directory: {Path.cwd()}")
        raise FileNotFoundError("ETH.db not found")

print(f"✅ Database found: {db_path}")

# Connect to database
conn = sqlite3.connect(str(db_path))
print("✅ Connected to database")

✅ Database found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\data\ETH.db
✅ Connected to database


### List Tables

In [3]:
# Get all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("=" * 50)
print("📋 TABLES IN DATABASE")
print("=" * 50)

if tables.empty:
    print("❌ No tables found!")
else:
    for _, row in tables.iterrows():
        table_name = row['name']
        # Get row count
        count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table_name};", conn).iloc[0, 0]
        print(f"  📊 {table_name:20s} : {count:>8,} rows")

# Check if eth_ohlcv exists
if 'eth_ohlcv' not in tables['name'].values:
    print("\n❌ Table 'eth_ohlcv' not found!")
    print("Available tables:", tables['name'].tolist())
    conn.close()
    raise ValueError("eth_ohlcv table not in database")

📋 TABLES IN DATABASE
  📊 eth_daily            :      365 rows
  📊 eth_signals          :      365 rows
  📊 trades               :        5 rows
  📊 sqlite_sequence      :        2 rows
  📊 performance          :        0 rows
  📊 signals              :       15 rows
  📊 eth_ohlcv            :      395 rows


### Load Data with Column Check

In [4]:
# First, check column names
sample = pd.read_sql("SELECT * FROM eth_ohlcv LIMIT 1;", conn)
print("📋 Available columns:")
print(sample.columns.tolist())

# Load all data
df = pd.read_sql("SELECT * FROM eth_ohlcv ORDER BY date;", conn)

# Check if 'date' column exists (might be 'Date' or 'timestamp')
date_col = None
for col in ['date', 'Date', 'timestamp', 'Timestamp', 'time']:
    if col in df.columns:
        date_col = col
        break

if date_col:
    df[date_col] = pd.to_datetime(df[date_col])
    df.set_index(date_col, inplace=True)
else:
    print("⚠️  No date column found, using integer index")

# Check for price columns
price_col = None
for col in ['close', 'Close', 'price', 'Price']:
    if col in df.columns:
        price_col = col
        break

if price_col is None:
    print("❌ No price column found!")
    print("Available columns:", df.columns.tolist())
    conn.close()
    raise ValueError("No price column found")

print(f"\n✅ Using price column: '{price_col}'")
print(f"✅ Loaded {len(df):,} rows")
print(f"📅 Date range: {df.index.min()} to {df.index.max()}" if date_col else "📅 No date column")

conn.close()

# Show sample
display(df.head())

📋 Available columns:
['date', 'open', 'high', 'low', 'close', 'volume', 'SMA_7', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'MACD', 'MACD_signal', 'MACD_histogram', 'RSI_14', 'BB_middle', 'BB_upper', 'BB_lower', 'ATR_14']

✅ Using price column: 'close'
✅ Loaded 395 rows
📅 Date range: 2025-07-29 00:00:00 to 2026-07-28 00:00:00


,open,high,low,close,volume,SMA_7,SMA_20,SMA_50,EMA_12,EMA_26,MACD,MACD_signal,MACD_histogram,RSI_14,BB_middle,BB_upper,BB_lower,ATR_14
date,,,,,,,,,,,,,,,,,,
2025-07-29,3799.00,3886.44,3716.04,3793.79,584210.4393,NaN,NaN,NaN,3793.790000,3793.790000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
2025-07-30,3793.79,3834.03,3677.65,3810.00,525424.9003,NaN,NaN,NaN,3796.283846,3794.990741,1.293105,0.258621,1.034484,NaN,NaN,NaN,NaN,NaN
2025-07-31,3810.00,3878.67,3684.33,3698.39,516088.4503,NaN,NaN,NaN,3781.223254,3787.835130,-6.611876,-1.115478,-5.496398,NaN,NaN,NaN,NaN,NaN
2025-08-01,3698.39,3724.02,3431.75,3488.20,783869.1517,NaN,NaN,NaN,3736.142754,3765.639935,-29.497182,-6.791819,-22.705363,NaN,NaN,NaN,NaN,NaN
2025-08-02,3488.21,3537.69,3368.29,3393.94,494195.7094,NaN,NaN,NaN,3683.496176,3738.106607,-54.610431,-16.355541,-38.254889,NaN,NaN,NaN,NaN,NaN


### Data Overview

In [5]:
print("=" * 50)
print("📊 DATA OVERVIEW")
print("=" * 50)

print(f"Total Records: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Column names: {df.columns.tolist()}")

# Check for nulls
print("\n🔍 Missing Values:")
null_counts = df.isnull().sum()
if null_counts.sum() > 0:
    for col, count in null_counts.items():
        if count > 0:
            pct = (count / len(df)) * 100
            print(f"  {col:20s} : {count:>6,} ({pct:>5.1f}%)")
else:
    print("  ✅ No missing values!")

# Check duplicates
duplicates = df.index.duplicated().sum() if date_col else 0
print(f"\nDuplicate timestamps: {duplicates}")

# Basic stats
print("\n📊 Basic Statistics:")
display(df.describe())

📊 DATA OVERVIEW
Total Records: 395
Columns: 18
Column names: ['open', 'high', 'low', 'close', 'volume', 'SMA_7', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'MACD', 'MACD_signal', 'MACD_histogram', 'RSI_14', 'BB_middle', 'BB_upper', 'BB_lower', 'ATR_14']

🔍 Missing Values:
  SMA_7                :     36 (  9.1%)
  SMA_20               :     49 ( 12.4%)
  SMA_50               :     79 ( 20.0%)
  EMA_12               :     30 (  7.6%)
  EMA_26               :     30 (  7.6%)
  MACD                 :     30 (  7.6%)
  MACD_signal          :     30 (  7.6%)
  MACD_histogram       :     30 (  7.6%)
  RSI_14               :     43 ( 10.9%)
  BB_middle            :     49 ( 12.4%)
  BB_upper             :     49 ( 12.4%)
  BB_lower             :     49 ( 12.4%)
  ATR_14               :     43 ( 10.9%)

Duplicate timestamps: 30

📊 Basic Statistics:


,open,high,low,close,volume,SMA_7,SMA_20,SMA_50,EMA_12,EMA_26,MACD,MACD_signal,MACD_histogram,RSI_14,BB_middle,BB_upper,BB_lower,ATR_14
count,395.000000,395.000000,395.000000,395.000000,3.950000e+02,359.000000,346.000000,316.000000,365.000000,365.000000,365.000000,365.000000,365.000000,352.000000,346.000000,346.000000,346.000000,352.000000
mean,2787.995038,2852.488354,2716.269443,2783.145013,4.046649e+05,2862.522642,2861.888386,2844.108396,2890.107533,2928.033711,-37.926179,-38.356148,0.429969,47.107936,2861.888386,3171.840862,2551.935909,146.364568
std,955.899833,983.000018,927.577915,955.616896,2.863877e+05,946.906726,934.113704,864.848900,936.334708,915.355395,99.767118,93.020338,31.631093,15.687701,934.113704,1047.317903,846.795635,66.712376
min,1567.860000,1588.820000,1505.680000,1567.840000,1.177100e+04,1583.875714,1669.393500,1732.783400,1624.304782,1695.881177,-277.939829,-253.062840,-103.685809,4.287692,1669.393500,1810.826348,1451.436069,59.489286
25%,1967.735000,2020.575000,1918.070000,1965.545000,2.083367e+05,2038.997857,2087.789750,2120.844000,2078.425147,2112.412675,-87.299503,-90.560888,-20.666328,34.285227,2087.789750,2297.720497,1849.420766,93.255357
50%,2350.330000,2394.710000,2300.550000,2348.170000,3.645353e+05,2872.938571,2726.885500,2653.848200,2962.668256,3004.752990,-32.543902,-31.017265,1.177192,48.026374,2726.885500,3199.047942,2250.584511,124.374286
75%,3492.480000,3635.760000,3369.970000,3462.125000,5.110799e+05,3653.617857,3713.310000,3426.064950,3746.986791,3793.790000,29.989657,24.733617,22.972901,59.694854,3713.310000,4217.916519,3163.111546,186.699643
max,4832.070000,4956.780000,4711.000000,4832.070000,2.102312e+06,4626.478571,4484.077500,4406.657600,4516.162441,4449.454562,205.877683,175.262787,112.198972,84.273261,4484.077500,5021.068056,4177.377409,317.668571


### Price Analysis 

In [6]:
print("=" * 50)
print("📈 PRICE ANALYSIS")
print("=" * 50)

if price_col:
    print(f"Current Price: {df[price_col].iloc[-1]:.4f}")
    print(f"Highest Price: {df[price_col].max():.4f}")
    print(f"Lowest Price: {df[price_col].min():.4f}")
    print(f"Average Price: {df[price_col].mean():.4f}")
    print(f"Price Std Dev: {df[price_col].std():.4f}")
    
    # Calculate returns
    df['returns'] = df[price_col].pct_change() * 100
    
    print(f"\n📊 Returns:")
    print(f"  Total Return: {((df[price_col].iloc[-1] / df[price_col].iloc[0] - 1) * 100):.2f}%")
    print(f"  Avg Daily Return: {df['returns'].mean():.4f}%")
    print(f"  Volatility: {df['returns'].std():.4f}%")
    print(f"  Best Day: {df['returns'].max():.4f}%")
    print(f"  Worst Day: {df['returns'].min():.4f}%")
    
    # Positive/Negative days
    positive = (df['returns'] > 0).sum()
    negative = (df['returns'] < 0).sum()
    print(f"\n  Positive Days: {positive} ({positive/len(df)*100:.1f}%)")
    print(f"  Negative Days: {negative} ({negative/len(df)*100:.1f}%)")

📈 PRICE ANALYSIS
Current Price: 1883.4300
Highest Price: 4832.0700
Lowest Price: 1567.8400
Average Price: 2783.1450
Price Std Dev: 955.6169

📊 Returns:
  Total Return: -50.35%
  Avg Daily Return: -0.1058%
  Volatility: 3.8009%
  Best Day: 14.3604%
  Worst Day: -14.9623%

  Positive Days: 190 (48.1%)
  Negative Days: 204 (51.6%)
